# Ejercicio 4. Pipeline declarativo

## Objetivo

Implementar un pipeline ETL con Lakeflow Spark Declarative Pipelines para limpiar y resumir las ventas de una tienda de videojuegos.

El pipeline utiliza la tabla `ventas_bronze` como fuente y genera dos vistas materializadas:

- `pipeline_ventas_limpias`: contiene las ventas válidas.
- `pipeline_ventas_por_categoria`: calcula el valor total vendido por categoría.

Finalmente, se agrega una nueva venta para comprobar que el pipeline actualiza sus resultados.


## 4.1. Preguntas de investigación

### ¿Qué es un pipeline de datos y cuáles son sus etapas habituales?

Un pipeline de datos es un proceso que sirve para llevar información desde una fuente hasta un destino, realizando diferentes operaciones sobre los datos.

Por lo general, primero se extraen los datos, después se limpian y transforman según lo que se necesite y finalmente se almacenan para poder utilizarlos en consultas, análisis o reportes.

### ¿Qué significa que un pipeline sea declarativo?

Significa que nosotros definimos qué resultados queremos obtener, mientras que la herramienta se encarga de organizar cómo se van a ejecutar las transformaciones y las dependencias que existen entre ellas.

### ¿En qué se diferencian un job que ejecuta notebooks y un pipeline declarativo?

Un job permite ejecutar diferentes tareas siguiendo un orden establecido. Por ejemplo, podemos configurar que primero se ejecute un notebook que cargue los datos y después otro que los transforme.

Ya en cuanto a un pipeline declarativo, nosotros definimos qué datos queremos generar y Databricks se encarga de administrar las dependencias y el orden en que se deben realizar las transformaciones.

### Fuentes consultadas

- [Lakeflow Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/)
- [Databricks Jobs](https://docs.databricks.com/aws/en/jobs/)
- [Vistas materializadas](https://docs.databricks.com/aws/en/ldp/materialized-views)


## 4.2. Resultados iniciales del pipeline

La primera ejecución del pipeline se completó correctamente.

Se obtuvieron 12 registros en la vista de ventas limpias y cuatro categorías en el resumen.

Antes de agregar una nueva venta, se consulta el total vendido por categoría para establecer un punto de comparación.


In [0]:
from pyspark.sql.functions import col

tabla_resumen = (
    "workspace.ingenieria_datos_luismiguel."
    "pipeline_ventas_por_categoria"
)

df_antes = spark.table(tabla_resumen)

display(df_antes.orderBy("categoria"))


categoria,total_vendido
Accesorios,4028500.0
Consolas,6999700.0
Preventas,699800.0
Videojuegos,2398900.0


## 4.3. Inserción de una nueva venta

Se agrega una venta válida de un videojuego a la tabla `ventas_bronze`.

La nueva venta tiene las siguientes características:

- **ID:** 15
- **Fecha:** 14 de noviembre de 2026
- **Producto:** The Legend of Zelda: Tears of the Kingdom
- **Categoría:** Videojuegos
- **Cantidad:** 2 unidades
- **Precio unitario:** $279.900 COP
- **Total:** $559.800 COP

Esta venta permitirá comprobar si el pipeline actualiza correctamente el resumen por categoría.


In [0]:
from datetime import date
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    StringType, DateType, DoubleType
)

tabla_bronze = (
    "workspace.ingenieria_datos_luismiguel.ventas_bronze"
)

schema = spark.table(tabla_bronze).schema

nueva_venta = [
    (
        15,
        date(2026, 11, 14),
        "The Legend of Zelda: Tears of the Kingdom",
        "Videojuegos",
        2,
        279900.0
    )
]

df_nueva = spark.createDataFrame(nueva_venta, schema)

# Evitar insertar dos veces la misma venta
if spark.table(tabla_bronze).filter(col("id_venta") == 15).count() == 0:
    df_nueva.write.format("delta").mode("append").saveAsTable(tabla_bronze)
    print("Nueva venta agregada correctamente")
else:
    print("La venta ya existe, no se duplicó")

print("Registros Bronze:", spark.table(tabla_bronze).count())


Nueva venta agregada correctamente
Registros Bronze: 16


## 4.4. Actualización del pipeline

Después de insertar una nueva venta válida en `ventas_bronze`, se ejecutó nuevamente el pipeline declarativo sin ejecutar el notebook de carga original.

La actualización se completó correctamente y se obtuvieron los siguientes resultados:

| Resultado | Antes | Después |
|---|---|---|
| Ventas limpias | 12 | 13 |
| Categorías | 4 | 4 |

El grafo de dependencias confirmó que la vista `pipeline_ventas_limpias` se actualizó de forma incremental.

Esto demuestra que el pipeline puede procesar cambios en los datos de origen y actualizar sus resultados.


In [0]:

from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

# Valores registrados antes de actualizar el pipeline
valores_iniciales = [
    ("Accesorios", 4028500.0),
    ("Consolas", 6999700.0),
    ("Preventas", 699800.0),
    ("Videojuegos", 2398900.0)
]

df_antes_fijo = spark.createDataFrame(
    valores_iniciales,
    ["categoria", "total_antes"]
)

# Consultar los resultados actualizados
tabla_resumen = (
    "workspace.ingenieria_datos_luismiguel."
    "pipeline_ventas_por_categoria"
)

df_despues = spark.table(tabla_resumen)

# Comparacion por categoria
comparacion = (
    df_antes_fijo
    .join(df_despues, "categoria", "inner")
    .select(
        "categoria",
        spark_round(col("total_antes"), 0)
            .cast("long").alias("total_antes"),
        spark_round(col("total_vendido"), 0)
            .cast("long").alias("total_despues"),
        spark_round(
            col("total_vendido") - col("total_antes"), 0
        ).cast("long").alias("diferencia")
    )
)

print("COMPARACION DE VENTAS POR CATEGORIA")
display(comparacion.orderBy("categoria"))

# Resumen general con formato de tabla
resumen_general = comparacion.agg(
    spark_sum("total_antes").alias("total_antes"),
    spark_sum("total_despues").alias("total_despues"),
    spark_sum("diferencia").alias("incremento")
).withColumn(
    "concepto", col("total_antes") * 0
).drop("concepto")

print("RESUMEN GENERAL DE VENTAS (COP)")
display(resumen_general)


COMPARACION DE VENTAS POR CATEGORIA


categoria,total_antes,total_despues,diferencia
Accesorios,4028500,4028500,0
Consolas,6999700,6999700,0
Preventas,699800,699800,0
Videojuegos,2398900,2958700,559800


RESUMEN GENERAL DE VENTAS (COP)


total_antes,total_despues,incremento
14126900,14686700,559800


## 4.5. Resultados y conclusión

### Resultados obtenidos

Se ejecutó correctamente el pipeline declarativo utilizando Lakeflow Spark Declarative Pipelines y cómputo Serverless.

El pipeline generó dos vistas materializadas:

- `pipeline_ventas_limpias`: elimina registros inválidos y duplicados.
- `pipeline_ventas_por_categoria`: calcula el total vendido por categoría.

Inicialmente se obtuvieron 12 ventas válidas y 4 categorías.

Posteriormente, se agregó una nueva venta de **The Legend of Zelda: Tears of the Kingdom**, correspondiente a 2 unidades de $279.900 COP cada una.

Al ejecutar nuevamente el pipeline, se obtuvieron los siguientes resultados:

| Indicador | Antes | Después |
|---|---:|---:|
| Ventas válidas | 12 | 13 |
| Categorías | 4 | 4 |
| Total vendido | $14.126.900 | $14.686.700 |
| Ventas de Videojuegos | $2.398.900 | $2.958.700 |

El incremento total fue de **$559.800 COP**, que coincide con el valor de la nueva venta.

### Conclusión

El ejercicio permitió comprobar el funcionamiento de un pipeline declarativo en Databricks.

Las vistas materializadas se actualizaron correctamente después de modificar los datos de origen, sin necesidad de ejecutar nuevamente el notebook de carga.

Además, el grafo mostró las dependencias entre las vistas y confirmó una actualización incremental de `pipeline_ventas_limpias`.

Esto demuestra que Lakeflow permite organizar transformaciones, administrar dependencias y mantener actualizados los resultados de un proceso de datos.
